# Cyber Threat Intelligence Platform

## 1. Carregamento do Dataset

Dataset utilizado: CICIDS2017, com tráfego de rede benigno e ataques DDoS.

In [22]:
from pathlib import Path
import urllib.request

dataset_path = Path("CICIDS2017_DDoS.parquet")

dataset_url = (
    "https://huggingface.co/datasets/bvsam/cic-ids-2017/resolve/main/"
    "machine_learning/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv.parquet"
)

if not dataset_path.exists():
    print("Baixando dataset...")
    urllib.request.urlretrieve(dataset_url, dataset_path)
    print("Dataset baixado com sucesso.")
else:
    print("Dataset já existe.")

Dataset já existe.


## 2. Leitura dos dados

In [2]:
import pandas as pd

df = pd.read_parquet(dataset_path)

print(df.shape)
df.head()

(225745, 79)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


## 3. Auditoria inicial dos dados

In [23]:
import numpy as np

print("Shape:", df.shape)

print("\nTipos:")
print(df.dtypes.value_counts())

print("\nValores ausentes:")
print(df.isna().sum().sum())

print("\nDuplicados:")
print(df.duplicated().sum())

print("\nClasses:")
print(df["Label"].value_counts())

print("\nProporção das classes:")
print(df["Label"].value_counts(normalize=True) * 100)

print("\nValores infinitos:")
numeric_df = df.select_dtypes(include="number")
print(np.isinf(numeric_df).sum().sum())

print("\nEstatísticas:")
display(numeric_df.describe().T)

Shape: (225745, 79)

Tipos:
int64      54
float64    24
str         1
Name: count, dtype: int64

Valores ausentes:
4

Duplicados:
2633

Classes:
Label
DDoS      128027
BENIGN     97718
Name: count, dtype: int64

Proporção das classes:
Label
DDoS      56.713105
BENIGN    43.286895
Name: proportion, dtype: float64

Valores infinitos:
64

Estatísticas:


c:\Users\andre\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
c:\Users\andre\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,count,mean,std,min,25%,50%,75%,max
Destination Port,225745.0,8.879619e+03,1.975465e+04,0.0,80.0,80.0,80.0,65532.0
Flow Duration,225745.0,1.624165e+07,3.152437e+07,-1.0,71180.0,1452333.0,8805237.0,119999937.0
Total Fwd Packets,225745.0,4.874916e+00,1.542287e+01,1.0,2.0,3.0,5.0,1932.0
Total Backward Packets,225745.0,4.572775e+00,2.175536e+01,0.0,1.0,4.0,5.0,2942.0
Total Length of Fwd Packets,225745.0,9.394633e+02,3.249403e+03,0.0,26.0,30.0,63.0,183012.0
...,...,...,...,...,...,...,...,...
Active Min,225745.0,1.776201e+05,7.842602e+05,0.0,0.0,0.0,1862.0,100000000.0
Idle Mean,225745.0,1.032214e+07,2.185303e+07,0.0,0.0,0.0,8239725.0,120000000.0
Idle Std,225745.0,3.611943e+06,1.275689e+07,0.0,0.0,0.0,0.0,65300000.0
Idle Max,225745.0,1.287813e+07,2.692126e+07,0.0,0.0,0.0,8253838.0,120000000.0


## 4. Limpeza dos dados

In [24]:
df_clean = df.copy()

df_clean = df_clean.replace([np.inf, -np.inf], np.nan)

print("Ausentes após converter infinitos:")
print(df_clean.isna().sum().sum())

df_clean = df_clean.dropna()

print("\nDuplicados antes:")
print(df_clean.duplicated().sum())

df_clean = df_clean.drop_duplicates()

print("\nShape final:")
print(df_clean.shape)

Ausentes após converter infinitos:
68

Duplicados antes:
2629

Shape final:
(223082, 79)


## 5. Validação após a limpeza

In [25]:
print("Shape:", df_clean.shape)

print("\nValores ausentes:")
print(df_clean.isna().sum().sum())

print("\nValores infinitos:")
numeric_clean = df_clean.select_dtypes(include="number")
print(np.isinf(numeric_clean).sum().sum())

print("\nDuplicados:")
print(df_clean.duplicated().sum())

print("\nClasses:")
print(df_clean["Label"].value_counts())

print("\nFlow Duration negativo:")
print((df_clean["Flow Duration"] < 0).sum())

print("\nDistribuição dos negativos por classe:")
print(df_clean.loc[df_clean["Flow Duration"] < 0, "Label"].value_counts())

Shape: (223082, 79)

Valores ausentes:
0

Valores infinitos:
0

Duplicados:
0

Classes:
Label
DDoS      128014
BENIGN     95068
Name: count, dtype: int64

Flow Duration negativo:
2

Distribuição dos negativos por classe:
Label
BENIGN    2
Name: count, dtype: int64


## 6. Construção da camada Silver

In [26]:
df_silver = df_clean[df_clean["Flow Duration"] >= 0].copy()

df_silver = df_silver.reset_index(drop=True)

print("Shape Silver:", df_silver.shape)

print("\nValores ausentes:")
print(df_silver.isna().sum().sum())

print("\nValores infinitos:")
numeric_silver = df_silver.select_dtypes(include="number")
print(np.isinf(numeric_silver).sum().sum())

print("\nDuplicados:")
print(df_silver.duplicated().sum())

print("\nClasses:")
print(df_silver["Label"].value_counts())

df_silver.to_parquet("CICIDS2017_DDoS_Silver.parquet", index=False)

Shape Silver: (223080, 79)

Valores ausentes:
0

Valores infinitos:
0

Duplicados:
0

Classes:
Label
DDoS      128014
BENIGN     95066
Name: count, dtype: int64


## 7. Preparação da camada Gold

In [27]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df_gold = df_silver.copy()

df_gold["target"] = df_gold["Label"].map({
    "BENIGN": 0,
    "DDoS": 1
})

X = df_gold.drop(columns=["Label", "target"])
y = df_gold["target"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

constant_columns = X_train.columns[X_train.nunique() <= 1]

X_train = X_train.drop(columns=constant_columns)
X_val = X_val.drop(columns=constant_columns)
X_test = X_test.drop(columns=constant_columns)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

## 8. Modelo baseline — Regressão Logística

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

logistic_model = LogisticRegression(max_iter=1000)

logistic_model.fit(X_train_scaled, y_train)

y_val_pred = logistic_model.predict(X_val_scaled)

print("Accuracy:", accuracy_score(y_val, y_val_pred))
print("Precision:", precision_score(y_val, y_val_pred))
print("Recall:", recall_score(y_val, y_val_pred))
print("F1-score:", f1_score(y_val, y_val_pred))

Accuracy: 0.9981770366385752
Precision: 0.997918942822954
Recall: 0.998906363920425
F1-score: 0.9984124092340525


## 9. Avaliação do baseline

In [9]:
from sklearn.metrics import confusion_matrix, classification_report

print("Matriz de confusão:")
print(confusion_matrix(y_val, y_val_pred))

print("\nRelatório:")
print(classification_report(
    y_val,
    y_val_pred,
    target_names=["BENIGN", "DDoS"]
))

Matriz de confusão:
[[14220    40]
 [   21 19181]]

Relatório:
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00     14260
        DDoS       1.00      1.00      1.00     19202

    accuracy                           1.00     33462
   macro avg       1.00      1.00      1.00     33462
weighted avg       1.00      1.00      1.00     33462



## 10. Avaliação no conjunto de teste

In [10]:
y_test_pred = logistic_model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("Precision:", precision_score(y_test, y_test_pred))
print("Recall:", recall_score(y_test, y_test_pred))
print("F1-score:", f1_score(y_test, y_test_pred))

print("\nMatriz de confusão:")
print(confusion_matrix(y_test, y_test_pred))

Accuracy: 0.9983264598649214
Precision: 0.9980749219562955
Recall: 0.9990105197375273
F1-score: 0.9985425016917391

Matriz de confusão:
[[14223    37]
 [   19 19183]]


## 11. Comparação com Random Forest

In [11]:
from sklearn.ensemble import RandomForestClassifier

random_forest_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

random_forest_model.fit(X_train, y_train)

y_val_pred_rf = random_forest_model.predict(X_val)

print("Accuracy:", accuracy_score(y_val, y_val_pred_rf))
print("Precision:", precision_score(y_val, y_val_pred_rf))
print("Recall:", recall_score(y_val, y_val_pred_rf))
print("F1-score:", f1_score(y_val, y_val_pred_rf))

print("\nMatriz de confusão:")
print(confusion_matrix(y_val, y_val_pred_rf))

Accuracy: 0.9999402307094615
Precision: 1.0
Recall: 0.9998958441828976
F1-score: 0.999947919379199

Matriz de confusão:
[[14260     0]
 [    2 19200]]


## 12. Avaliação final do Random Forest

In [12]:
y_test_pred_rf = random_forest_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_test_pred_rf))
print("Precision:", precision_score(y_test, y_test_pred_rf))
print("Recall:", recall_score(y_test, y_test_pred_rf))
print("F1-score:", f1_score(y_test, y_test_pred_rf))

print("\nMatriz de confusão:")
print(confusion_matrix(y_test, y_test_pred_rf))

Accuracy: 0.999880461418923
Precision: 0.9999479166666667
Recall: 0.9998437662743465
F1-score: 0.999895838758398

Matriz de confusão:
[[14259     1]
 [    3 19199]]


## 13. Importância das variáveis e auditoria de leakage

In [13]:
import pandas as pd

feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": random_forest_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

display(feature_importance.head(20))

,Feature,Importance
49,Avg Fwd Segment Size,0.115924
8,Fwd Packet Length Mean,0.083666
6,Fwd Packet Length Max,0.074996
22,Fwd IAT Std,0.070625
56,Init_Win_bytes_forward,0.066786
0,Destination Port,0.055543
53,Subflow Fwd Bytes,0.054371
4,Total Length of Fwd Packets,0.051374
58,act_data_pkt_fwd,0.048267
20,Fwd IAT Total,0.044094


## 14. Auditoria de redundância entre variáveis

In [14]:
suspected_features = [
    "Avg Fwd Segment Size",
    "Fwd Packet Length Mean",
    "Subflow Fwd Bytes",
    "Total Length of Fwd Packets",
    "Subflow Fwd Packets",
    "Total Fwd Packets",
    "Fwd Header Length_duplicated_0",
    "Fwd Header Length"
]

correlation_matrix = df_gold[suspected_features].corr()

display(correlation_matrix.round(4))

,Avg Fwd Segment Size,Fwd Packet Length Mean,Subflow Fwd Bytes,Total Length of Fwd Packets,Subflow Fwd Packets,Total Fwd Packets,Fwd Header Length_duplicated_0,Fwd Header Length
Avg Fwd Segment Size,1.0000,1.0000,0.8583,0.8583,0.0169,0.0169,0.0169,0.0169
Fwd Packet Length Mean,1.0000,1.0000,0.8583,0.8583,0.0169,0.0169,0.0169,0.0169
Subflow Fwd Bytes,0.8583,0.8583,1.0000,1.0000,0.1739,0.1739,0.1760,0.1760
Total Length of Fwd Packets,0.8583,0.8583,1.0000,1.0000,0.1739,0.1739,0.1760,0.1760
Subflow Fwd Packets,0.0169,0.0169,0.1739,0.1739,1.0000,1.0000,0.9684,0.9684
Total Fwd Packets,0.0169,0.0169,0.1739,0.1739,1.0000,1.0000,0.9684,0.9684
Fwd Header Length_duplicated_0,0.0169,0.0169,0.1760,0.1760,0.9684,0.9684,1.0000,1.0000
Fwd Header Length,0.0169,0.0169,0.1760,0.1760,0.9684,0.9684,1.0000,1.0000


## 15. Remoção de variáveis redundantes

In [15]:
redundant_columns = [
    "Avg Fwd Segment Size",
    "Subflow Fwd Bytes",
    "Subflow Fwd Packets",
    "Fwd Header Length_duplicated_0"
]

X_train_reduced = X_train.drop(columns=redundant_columns)
X_val_reduced = X_val.drop(columns=redundant_columns)
X_test_reduced = X_test.drop(columns=redundant_columns)

print("Antes:", X_train.shape)
print("Depois:", X_train_reduced.shape)

Antes: (156156, 68)
Depois: (156156, 64)


In [16]:
random_forest_reduced = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

random_forest_reduced.fit(X_train_reduced, y_train)

y_val_pred_rf_reduced = random_forest_reduced.predict(X_val_reduced)

print("Accuracy:", accuracy_score(y_val, y_val_pred_rf_reduced))
print("Precision:", precision_score(y_val, y_val_pred_rf_reduced))
print("Recall:", recall_score(y_val, y_val_pred_rf_reduced))
print("F1-score:", f1_score(y_val, y_val_pred_rf_reduced))

print("\nMatriz de confusão:")
print(confusion_matrix(y_val, y_val_pred_rf_reduced))

Accuracy: 0.9999402307094615
Precision: 1.0
Recall: 0.9998958441828976
F1-score: 0.999947919379199

Matriz de confusão:
[[14260     0]
 [    2 19200]]


In [17]:
y_test_pred_rf_reduced = random_forest_reduced.predict(X_test_reduced)

print("Accuracy:", accuracy_score(y_test, y_test_pred_rf_reduced))
print("Precision:", precision_score(y_test, y_test_pred_rf_reduced))
print("Recall:", recall_score(y_test, y_test_pred_rf_reduced))
print("F1-score:", f1_score(y_test, y_test_pred_rf_reduced))

print("\nMatriz de confusão:")
print(confusion_matrix(y_test, y_test_pred_rf_reduced))

Accuracy: 0.999880461418923
Precision: 0.9999479166666667
Recall: 0.9998437662743465
F1-score: 0.999895838758398

Matriz de confusão:
[[14259     1]
 [    3 19199]]


## 16. Salvamento do modelo final

In [18]:
import joblib

joblib.dump(
    random_forest_reduced,
    "random_forest_ddos_model.pkl"
)

joblib.dump(
    X_train_reduced.columns.tolist(),
    "model_features.pkl"
)

print("Modelo salvo com sucesso.")
print("Número de features:", len(X_train_reduced.columns))

Modelo salvo com sucesso.
Número de features: 64


## 17. Função de predição

In [19]:
import joblib
import pandas as pd

loaded_model = joblib.load("random_forest_ddos_model.pkl")
model_features = joblib.load("model_features.pkl")

def predict_ddos(input_data):
    input_df = pd.DataFrame([input_data])
    input_df = input_df[model_features]

    prediction = loaded_model.predict(input_df)[0]
    probability = loaded_model.predict_proba(input_df)[0]

    return {
        "prediction": int(prediction),
        "class": "DDoS" if prediction == 1 else "BENIGN",
        "probability_benign": float(probability[0]),
        "probability_ddos": float(probability[1])
    }

## 18. Teste da função de predição

In [20]:
sample = X_test_reduced.iloc[0].to_dict()

result = predict_ddos(sample)

print("Resultado da função:")
print(result)

print("\nClasse real:")
print(int(y_test.iloc[0]))

Resultado da função:
{'prediction': 1, 'class': 'DDoS', 'probability_benign': 0.0, 'probability_ddos': 1.0}

Classe real:
1


## 19. Amostra DDoS real para teste da API

In [21]:
ddos_index = y_test[y_test == 1].index[0]

ddos_sample = X_test_reduced.loc[ddos_index].to_dict()

print("Classe real:", int(y_test.loc[ddos_index]))
print("\nAmostra DDoS real:")
print(ddos_sample)

Classe real: 1

Amostra DDoS real:
{'Destination Port': 80.0, 'Flow Duration': 30804.0, 'Total Fwd Packets': 3.0, 'Total Backward Packets': 6.0, 'Total Length of Fwd Packets': 26.0, 'Total Length of Bwd Packets': 11601.0, 'Fwd Packet Length Max': 20.0, 'Fwd Packet Length Min': 0.0, 'Fwd Packet Length Mean': 8.666666667, 'Fwd Packet Length Std': 10.26320288, 'Bwd Packet Length Max': 5840.0, 'Bwd Packet Length Min': 0.0, 'Bwd Packet Length Mean': 1933.5, 'Bwd Packet Length Std': 2189.772933, 'Flow Bytes/s': 377450.9804, 'Flow Packets/s': 292.1698481, 'Flow IAT Mean': 3850.5, 'Flow IAT Std': 10506.07246, 'Flow IAT Max': 29850.0, 'Flow IAT Min': 3.0, 'Fwd IAT Total': 30037.0, 'Fwd IAT Mean': 15018.5, 'Fwd IAT Std': 21235.12375, 'Fwd IAT Max': 30034.0, 'Fwd IAT Min': 3.0, 'Bwd IAT Total': 30620.0, 'Bwd IAT Mean': 6124.0, 'Bwd IAT Std': 13325.45138, 'Bwd IAT Max': 29960.0, 'Bwd IAT Min': 45.0, 'Fwd PSH Flags': 0.0, 'Fwd Header Length': 72.0, 'Bwd Header Length': 132.0, 'Fwd Packets/s': 97.38